# When Sources Disagree on Speaker Counts

CLDR, CIA, and LinguaMeta don't always agree on how many people speak a language
in a given country. This notebook compares per-country speaker estimates across
sources using `db.speaker_counts`.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)
for source in ("cldr", "cia", "linguameta", "scraped"):
    print(f"  {source}: {len(db.speaker_counts.by_source(source)):,} records")

## 2 · Query by source, country, and language

In [ ]:
rw = db.countries.get("RW")
print(f"{rw.label} population: {rw.population:,}")
print()
for sc in rw.speaker_counts:
    print(f"  {sc.language.label}: {sc.speaker_count:,} ({sc.speaker_fraction:.1%}) [{sc.source}]")

print()
kin = db.languages.get("kin")
print(f"{kin.label} — per-country breakdown:")
for sc in kin.speaker_counts:
    print(f"  {sc.country.label}: {sc.speaker_count:,} ({sc.source})")

## 3 · Build a comparison table for CLDR vs CIA

Pair records that share the same `(country, language)` key.

In [ ]:
def build_pairs(source_a, source_b):
    lookup = {}
    for sc in db.speaker_counts.by_source(source_a):
        key = (sc.country.code, sc.language.part3)
        lookup.setdefault(key, {})[source_a] = sc
    rows = []
    for sc in db.speaker_counts.by_source(source_b):
        key = (sc.country.code, sc.language.part3)
        if key not in lookup:
            continue
        a = lookup[key][source_a]
        rows.append({
            "country": sc.country.label,
            "country_code": sc.country.code,
            "language": sc.language.label,
            "part3": sc.language.part3,
            f"{source_a}_count": a.speaker_count,
            f"{source_a}_fraction": a.speaker_fraction,
            f"{source_b}_count": sc.speaker_count,
            f"{source_b}_fraction": sc.speaker_fraction,
            "population": sc.country.population,
        })
    return pd.DataFrame(rows)

pairs_df = build_pairs("cldr", "cia")
print(f"Country-language pairs in both CLDR and CIA: {len(pairs_df)}")
pairs_df.head(10)

## 4 · Scatter plot — CLDR vs CIA counts

In [ ]:
fig = px.scatter(
    pairs_df,
    x="cldr_count",
    y="cia_count",
    hover_name="language",
    hover_data={"country": True, "cldr_count": ":,", "cia_count": ":,"},
    log_x=True,
    log_y=True,
    labels={"cldr_count": "CLDR speaker count", "cia_count": "CIA speaker count"},
    title="CLDR vs CIA Speaker Counts (log-log)",
)
max_val = max(pairs_df["cldr_count"].max(), pairs_df["cia_count"].max())
fig.add_shape(type="line", x0=1, y0=1, x1=max_val, y1=max_val,
              line=dict(dash="dash", color="gray"))
fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · Relative disagreement

In [ ]:
pairs_df["ratio"] = pairs_df.apply(
    lambda r: max(r["cldr_count"], r["cia_count"]) / max(min(r["cldr_count"], r["cia_count"]), 1),
    axis=1,
)

fig2 = px.histogram(
    pairs_df,
    x="ratio",
    nbins=30,
    labels={"ratio": "Max/min count ratio"},
    title="Distribution of CLDR/CIA Disagreement (ratio > 1 means sources differ)",
)
fig2.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig2.show()

print("Largest disagreements:")
print(pairs_df.nlargest(10, "ratio")[["country", "language", "cldr_count", "cia_count", "ratio"]].to_string(index=False))

## 6 · Collection helpers

In [ ]:
de_entries = db.speaker_counts.for_country("DE")
print(f"Speaker count records for Germany: {len(de_entries)}")
for sc in de_entries[:6]:
    print(f"  {sc.language.label}: {sc.speaker_count:,} ({sc.source})")

print()
deutsch = db.speaker_counts.for_language("deu")
print(f"Speaker count records for German worldwide: {len(deutsch)}")
for sc in deutsch[:6]:
    print(f"  {sc.country.label}: {sc.speaker_count:,} ({sc.source})")

## 7 · Summary

In [ ]:
print(f"Pairs compared: {len(pairs_df)}")
print(f"Median disagreement ratio: {pairs_df['ratio'].median():.2f}")
print(f"Pairs where ratio > 2: {(pairs_df['ratio'] > 2).sum()}")